# 第10章 学習のお医者さん：うまく学習させる技

初期化・正規化・Adam・ドロップアウト・学習率スケジュールを救急箱に

## このノートブックの使い方・学習目標

- **教科書のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/10-training-tips.html
- **エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **復習・理解度管理**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html)
- **この章でつくるもの**：過学習を抑え、安定して学習する改良版MLPと、学習の「診察表」。モデルをGoogleドライブに保存する方法も
- **身につく数学・理論**：平均・分散・標準偏差と正規分布 ／ 初期化：最初の損失は $\log V$ 付近が健康 ／ 活性化の大きさが層を通るたびに暴れる問題（勾配消失・爆発） ／ LayerNorm（層正規化） ／ Adam / AdamW（慣性と、パラメータごとの歩幅） ／ 過学習とドロップアウト・重み減衰 ／ 学習率のウォームアップとコサイン減衰 ／ 勾配クリッピング
- **新しく使うPython・PyTorch**：`torch.randn` と `.mean()` `.std()` ／ `model.train()` と `model.eval()` ／ `torch.save` / `torch.load` ／ `google.colab.drive` でドライブに保存

## 実行環境・リソースの目安

| 項目 | 目安 |
|---|---|
| 推奨ランタイム | Google Colab（CPUでOK） |
| 所要時間目安 | 2〜3時間（2回に分けてOK） |
| 難易度 | ★★★☆☆ |

> **✅ CPUでそのまま実行できます**：上のセルから順に再生ボタン（▶）または `Shift + Enter` で実行してください。

> **🎯 この章でつくるもの**
>
> 第9章のMLPは、bigramの記録を破りましたが、途中から**丸暗記（過学習）**を始めていました。この章では、学習がうまくいかないときの「症状」と「治療法」をまとめて学びます。治療をすべて施した改良版MLPで、記録を **3.395 → 3.18 前後** まで伸ばします。ここで学ぶ技は、第13章のGPTでもそのまま使います。

> **🔗 これまでの復習とつながり：第5章・第7章・第9章とのつながり**
>
> - **第5章（訓練／検証の分割・初期損失 $\log V$）**：初見データ（検証）で本当の力を測ること、何も学んでいないモデルの損失は $\log V$ であることを学びました。この章では $\log V$ を「学習開始前の健康診断」に使います。
> - **第7章（学習率と勾配）**：学習率が大きすぎると発散し、小さすぎると進まないことを見ました。この章では AdamW・ウォームアップ・コサイン減衰・勾配クリッピングで坂の下り方を一気にプロ仕様へ強化します。
> - **第9章（MLPと過学習の芽）**：直前8文字のMLPで記録を更新した一方、訓練 `2.734` vs 検証 `3.395` と過学習が始まっていました。その治療薬（LayerNorm・ドロップアウト・重み減衰）を揃えます！

**ミナト**：「学習のお医者さん」って、なんだか大げさですね。

**ボンゴ先生**：いや、大げさじゃない。ニューラルネットワークの学習は、とても繊細な生き物だ。熱が出たり（損失が爆発）、寝込んだり（損失が下がらない）、物覚えが良すぎて応用がきかなくなったり（過学習）する。プロの研究者も、時間の大半を「診察」に使っている。今日はキミに聴診器を渡そう。

## 準備

準備セルは第9章とほぼ同じです。モデルには、この章で学ぶ部品（`LayerNorm` と `Dropout`）をあらかじめ入れてあります。説明は後で出てくるので、今は貼り付けて実行してください。

> **✨ コツ：2回に分けるときは**
>
> この章は長いので、「診察3：過学習」の手前で一度休むのがおすすめです。Colabは、しばらく放っておくとセッションが切れて、変数（`train` や `hist_a` など）が全部消えます。2回目は、メニューの「ランタイム」→「前のセルを実行」などで、**この準備セルから順に全部実行し直して**から続きを読みましょう（CPUで10分ほどかかります）。

🔎 **実行前に確認**：コードの各行が何を受け取り、何を作るか確認してから ▶ を押してください。

In [ ]:
import re, math, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F

def load_soseki(file_id="773_ruby_5968"):
    url = ("https://raw.githubusercontent.com/aozorahack/aozorabunko_text/0984f7dc18a2270bdfd2ff99bf14fd1b3ebb4f65/"
           f"cards/000148/files/{file_id}/{file_id}.txt")
    try:
        with urllib.request.urlopen(url, timeout=30) as res:
            text = res.read().decode("cp932")
    except Exception as e:
        raise RuntimeError(f"青空文庫の取得に失敗しました（時間をおいて再実行するか付録Eを参照）: {e}") from e
    text = text.split("-------------------------------------------------------")[2]
    text = text.split("底本：")[0]
    text = re.sub(r"《.*?》", "", text)
    text = re.sub(r"※?［＃.*?］", "", text)
    text = text.replace("｜", "").replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

text = load_soseki()
chars = sorted(set(text))
V = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text])
n = int(len(data) * 0.9)
train_data, val_data = data[:n], data[n:]
T = 8

class MLP(nn.Module):
    def __init__(self, V, T, d=32, H=256, dropout=0.0):
        super().__init__()
        self.emb = nn.Embedding(V, d)
        self.fc1 = nn.Linear(T * d, H)
        self.ln = nn.LayerNorm(H)           # この章で学ぶ「層正規化」
        self.drop = nn.Dropout(dropout)     # この章で学ぶ「ドロップアウト」
        self.fc2 = nn.Linear(H, V)

    def forward(self, x):
        e = self.emb(x)
        h = torch.tanh(self.ln(self.fc1(e.view(len(x), -1))))
        return self.fc2(self.drop(h))

val_X = torch.stack([val_data[i:i + T] for i in range(len(val_data) - T)])
val_Y = val_data[T:]

@torch.no_grad()
def val_loss(model):
    model.eval()
    loss = F.cross_entropy(model(val_X), val_Y).item()
    model.train()
    return loss

## 診察1：最初の損失を見る

学習を始める前に、まず **最初の損失** を見ます。何も学んでいないモデルは、全部の文字を同じくらいの確率で選ぶはずなので、損失は $\log V$ くらいになるはずでした（第8章）。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
torch.manual_seed(0)
model = MLP(V, T)
x, y = val_X[:512], val_Y[:512]
print("理想の最初の損失 log V：", round(math.log(V), 3))
print("このモデルの最初の損失：", round(F.cross_entropy(model(x), y).item(), 3))

with torch.no_grad():
    model.fc2.weight *= 30          # わざと出力層の重みを30倍にしてみる
print("重みを30倍にしたら：", round(F.cross_entropy(model(x), y).item(), 3))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
理想の最初の損失 log V： 7.631
このモデルの最初の損失： 7.7
重みを30倍にしたら： 37.845
```

</details>

重みを大きくしたら、最初の損失が30以上に跳ね上がりました。何も知らないくせに、点数（ロジット）に大きな差をつけて「この文字に違いない！」と**自信満々に間違えている**状態です。こうなると、学習の最初の時間を「自信過剰を直す」ことだけに使ってしまいます。

> **✨ コツ：処方箋：最初の損失は log V の近くに**
>
> 学習を始める前に、必ず最初の損失を `print` しましょう。$\log V$ から大きく外れていたら、初期化（パラメータの最初の値）がおかしいサインです。出力層の重みを小さめにしておくのが定石です。

#### 📝 確認問題（`ch10-q01` · Lv.2 · 初期損失と log V の比較による初期化診断）

語彙数 $V = 2061$（$\log V \approx 7.63$）のモデルを作った直後に最初の損失を測ったら `37.8` と極端に大きくなっていました。モデル内部で何が起きている可能性が最も高いですか？

- **a.** 訓練データと検証データの分割割合を 9:1 にしたことで検証サンプルが偏り、初期状態の平均損失が 5 倍近くに跳ね上がっている
- **b.** 初期損失は大きいほどこれからたくさん学習して改善できる余地が残っている証拠であり、むしろ非常に理想的な初期状態である
- **c.** 出力層の初期重みが大きすぎて、何も学んでいないのに特定の文字へ極端に高いロジットを付け、「自信満々に間違えている」状態になっている
- **d.** 出力層の重みとバイアスをすべて 0 で初期化したせいで、どの文字の予測確率も 0 に近づき初期損失が極端に大きくなっている

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：ランダムに選んだ 1 文字が極端に低い確率（例：`0.00000001`）しか与えられていないとき、交差エントロピー $-\log(p)$ がどう振る舞うか考えましょう。
- **ヒント 2**：出力層の初期重みが大きすぎると初期ロジットが極端にばらつき、正解以外の文字に 100% 近い確率を付けて「自信満々に外す」状態になります。
- **正解**：`c`

**考え方**：学習を始める前に「初期損失 $\approx \log V$」を必ずチェックするのは、初期化ミスやバグを最初の1秒で見抜くプロの習慣です。

1. 理想（まっさらな均等予測）：全文字の確率 $\approx 1/V$ → 損失 $\approx \log V = 7.631$。
1. 重みを30倍にしてロジットが暴れる → 間違った文字に自信満々、正解文字の確率が極小 → $-\log(p_{\text{正解}})$ が 37.8 に爆発。
1. 処方箋：出力層の重みを小さめに初期化し、初期損失が $\log V$ 付近になることを確認する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/10-training-tips.html#ex-ch10-q01

</details>

## 統計の道具：平均・分散・標準偏差

初期化の話をもう少し深めるために、統計の道具を3つだけ用意します。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
z = torch.randn(10000)       # 「標準正規分布」から10000個の数を引く
print("平均：", round(z.mean().item(), 3))
print("分散：", round(z.var().item(), 3))
print("標準偏差：", round(z.std().item(), 3))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
平均： -0.03
分散： 1.001
標準偏差： 1.001
```

</details>

#### 📐 数学のポイント：平均 $\mu$・分散 $\sigma^2$・標準偏差 $\sigma$：9ステップで理由から理解する

**① 解決したい問題**：ニューラルネットワークの中を流れる何千個もの数字について、「どのあたりを中心にして（位置）」「どれくらい広く暴れているか（ばらつき）」をたった2つの数で把握したい。

**② 素朴な方法では困る理由**：中心は平均で分かりますが、ばらつきを知ろうとして「平均からのずれ $(x_i - \mu)$」をそのまま足すと、**プラスのずれとマイナスのずれが必ず打ち消し合ってちょうど 0** になってしまいます！

**③ 新しい発想**：ずれを **2乗** してから平均する（＝**分散** $\sigma^2$）。すると0以上になりばらつきが測れます。ただし2乗すると単位が変わる（例：点 → $\text{点}^2$）ので、最後に平方根 $\sqrt{\cdot}$ で元の単位に戻したものが **標準偏差** $\sigma$ です。

$$\mu = \frac{1}{N}\sum_{i=1}^{N} x_i, \qquad \sigma^2 = \frac{1}{N}\sum_{i=1}^{N} (x_i - \mu)^2, \qquad \sigma = \sqrt{\sigma^2} = \sqrt{\frac{1}{N}\sum_{i=1}^{N} (x_i - \mu)^2}$$

**④ 記号の読み方テーブル**：

| 記号 | 読み方 | 意味 |
|---|---|---|
| $\mu$（または $\bar{x}$） | ミュー（エックスバー） | **平均**（mean）。データの真ん中 |
| $\sigma^2$ | シグマ2乗 | **分散**（variance）。平均からのずれの2乗の平均 |
| $\sigma$ | シグマ（小文字） | **標準偏差**（standard deviation）。分散の平方根（元の単位のばらつき） |
| $N$ | エヌ | データの個数（$\sum$ は第3章の `for` 文と同じ！） |

**⑤ 小さな3数で手計算**：`x = [2, 4, 6]` のとき：
1. **平均**：$\mu = (2 + 4 + 6) / 3 = 4$
2. **ずれ**：$2 - 4 = -2$、$4 - 4 = 0$、$6 - 4 = +2$（そのまま足すと $-2 + 0 + 2 = 0$ で消える！）
3. **分散**：$\sigma^2 = \big((-2)^2 + 0^2 + 2^2\big) / 3 = (4 + 0 + 4) / 3 = 8/3 \approx 2.667$
4. **標準偏差**：$\sigma = \sqrt{8/3} \approx 1.633$

**⑥ Pythonの最小コードで確認**：
```python no-run
import math
xs = [2, 4, 6]
mu = sum(xs) / len(xs)                               # 4.0
var = sum((x - mu) ** 2 for x in xs) / len(xs)       # 2.6666...
std = math.sqrt(var)                                  # 1.6329...
print(f"平均={mu:.3f}, 分散={var:.3f}, 標準偏差={std:.3f}")
```

`torch.randn` は **正規分布**（ガウス分布）という、平均0・標準偏差1の釣り鐘型のばらつきから数を引きます。身長やテストの点数など、自然界のいろいろなばらつきがこの形になります。ヒストグラムで見てみましょう。

📦 **準備セル**：ライブラリや教材ファイルを取得します。エラーが出た場合は `-q` を外して詳細を確認してください。

In [ ]:
!pip install -q matplotlib-fontja

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib_fontja

plt.figure(figsize=(6, 3))
plt.hist(z.numpy(), bins=60)
plt.title("torch.randn の数のばらつき（正規分布）")
plt.xlabel("値"); plt.ylabel("個数")
plt.show()

![正規分布のヒストグラム。0を中心にした釣り鐘型](https://sakunokyumei.github.io/llm-research-textbook/img/figs/10-training-tips/cell4_0.png)

#### 📝 確認問題（`ch10-q02` · Lv.1 · 平均 μ・分散 σ^2・標準偏差 σ の手計算）

3個のデータ `x = [1, 4, 7]` について、母分散 $\sigma^2 = \dfrac{1}{N} \sum_{i=1}^{N} (x_i - \mu)^2$ を手計算してください。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：まず3つの数の平均 $\mu$ を求め、各データから $\mu$ を引いた「ずれ（偏差）」をそれぞれ2乗しましょう。
- **ヒント 2**：平均は $\mu = (1+4+7)/3 = 4$、ずれは `[-3, 0, +3]` です。2乗した `[9, 0, 9]` の合計をデータ数 `3` で割ります。
- **正解**：`6`

**考え方**：分散は「平均からのずれを2乗したものの平均」です。$[1, 4, 7]$ では $(9 + 0 + 9) / 3 = 18 / 3 = 6$ になります。

1. 平均：$\mu = (1 + 4 + 7) / 3 = 4$。
1. ずれ（偏差）：$1-4 = -3$、$4-4 = 0$、$7-4 = +3$。
1. ずれの2乗：$(-3)^2 = 9$、$0^2 = 0$、$3^2 = 9$。
1. 分散：$\sigma^2 = (9 + 0 + 9) / 3 = 18 / 3 = 6$。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/10-training-tips.html#ex-ch10-q02

</details>

#### 📝 確認問題（`ch10-q03` · Lv.2 · 記号 μ・σ^2・σ の意味と PyTorch の unbiased=False）

分散の定義 $\sigma^2 = \dfrac{1}{N} \sum_{i=1}^{N} (x_i - \mu)^2$ で、$(x_i - \mu)$ をそのまま平均せずに**2乗してから平均し、最後に平方根をとって標準偏差 $\sigma$ にする**理由として正しいものはどれですか？

- **a.** 各要素のずれを 2 乗してから平方根をとるとデータの個数 $N$ が半分に圧縮され、ニューラルネットワークの計算速度が 2 倍になるため
- **b.** データの中心である平均 $\mu$ がそのままでは正や負に偏ってしまうため、2 乗と平方根を通して平均 $\mu$ を必ず `0` にそろえるため
- **c.** そのまま足すとプラスとマイナスのずれが必ず打ち消し合って 0 になるため 2 乗してばらつきを測り（分散）、2 乗で変わった単位を戻すため平方根をとる（標準偏差）
- **d.** どんなデータを入れても分散の値が常に `1.0` に固定されるように調整し、異なるデータセット間で平均を直接比較できるようにするため

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：たとえば `[2, 4, 6]` の平均は `4`、平均からのずれは `[-2, 0, +2]` です。これを 2 乗せずにそのまま足すと何が起きるでしょうか？
- **ヒント 2**：ずれの和 $\sum (x_i - \mu)$ は常に 0 になるため 2 乗して正の値で平均（分散）をとります。さらに 2 乗で変わった単位を元に戻すのが平方根（標準偏差）です。
- **正解**：`c`

**考え方**：平均の定義上、$\sum (x_i - \mu)$ はどんなデータでも必ずちょうど 0 になってしまいます。2乗で正にしてから平均（分散 $\sigma^2$）をとり、平方根で元のスケールに戻したものが標準偏差 $\sigma$ です。

1. ずれの和：$(-2) + 0 + (+2) = 0$（どんなデータでも必ず 0！）。
1. 2乗の平均（分散 $\sigma^2$）：ばらつきが測れるが、単位が「元の単位の2乗（例：$\text{cm}^2$）」になる。
1. 平方根（標準偏差 $\sigma$）：元の単位（例：$\text{cm}$）に戻る。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/10-training-tips.html#ex-ch10-q03

</details>

*(この節の追加問題 `ch10-q04` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/10-training-tips.html で自動採点・復習できます)*

## 診察2：層を通るたびに暴れる数

ニューラルネットワークを深く（層をたくさん）すると、層を通るたびに数の大きさが変わっていきます。10層の全結合層＋tanhに、標準偏差1の数を流して、各層の出力の標準偏差を測ってみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def stack_stats(weight_scale, layers=10, width=256):
    torch.manual_seed(0)
    h = torch.randn(1000, width)
    stds = []
    for _ in range(layers):
        W = torch.randn(width, width) * weight_scale
        h = torch.tanh(h @ W)
        stds.append(round(h.std().item(), 3))
    return stds

print("重みの大きさ 1.0  ：", stack_stats(1.0))
print("重みの大きさ 0.01 ：", stack_stats(0.01))
print("重みの大きさ 1/√256：", stack_stats(1 / math.sqrt(256)))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
重みの大きさ 1.0  ： [0.975, 0.974, 0.974, 0.974, 0.974, 0.974, 0.974, 0.974, 0.974, 0.974]
重みの大きさ 0.01 ： [0.156, 0.025, 0.004, 0.001, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
重みの大きさ 1/√256： [0.628, 0.486, 0.408, 0.359, 0.322, 0.293, 0.271, 0.253, 0.238, 0.226]
```

</details>

> **💡 理論のポイント：勾配消失と飽和**
>
> - **重みが大きすぎる（1.0）**：数がどんどん大きくなり、tanhの出力がほとんど ±1 に張り付きます（標準偏差0.97≒ほぼ全部が±1）。tanhは±1の近くで**傾きがほぼ0**なので、誤差逆伝播で勾配がほとんど伝わりません（**飽和**）
> - **重みが小さすぎる（0.01）**：数がどんどん小さくなり、4層目ではほぼ0。信号が消えてしまいます。勾配も同じように消えていきます（**勾配消失**）
> - **ちょうどいい（$1/\sqrt{\text{入力の数}}$）**：大きさがゆっくり減る程度で、深い層まで信号が届きます
>
> 入力が256個あると、256個の数を足し合わせるので、ばらつきが $\sqrt{256} = 16$ 倍に膨らみます。だから重みを $1/\sqrt{256}$ にしておくと、ちょうど打ち消し合うのです。PyTorchの `nn.Linear` は、最初からこの考え方に基づいた大きさで初期化されています（**Xavier初期化**、**Kaiming初期化**などと呼ばれる方法の仲間）。
>
> 逆向きの勾配が層を通るたびに大きくなりすぎる場合は **勾配爆発** と言います。損失が突然 `nan`（計算不能）になるのは、たいていこれです。

#### 📝 確認問題（`ch10-q05` · Lv.2 · tanhの飽和・勾配消失と 1/√d_in 初期化の原理）

10層の `tanh` ネットワークで重みの初期スケールを変えたとき、「重みが大きすぎる（1.0）」場合と「小さすぎる（0.01）」場合に起きる問題の組み合わせとして**正しい**のはどれですか？

- **a.** 大きすぎると層を通るたびに数値が打ち消し合って信号が 0 へ消え（**勾配消失**）、小さすぎると出力がすべて ±1 の端に張り付いて動かなくなる（**飽和**）
- **b.** 重みの初期スケールが大きくても小さくても、`tanh` を 10 層重ねるうちに自動的に平均 0・標準偏差 1.0 の理想的な分布へ収束するので学習に問題はない
- **c.** 重みの初期値は学習ループの 1 ステップ目の勾配降下ですべて最適な値に上書きされるため、最初に大きすぎても小さすぎても最終性能にはまったく影響しない
- **d.** 大きすぎると `tanh` の出力が ±1 に張り付いて傾きがほぼ 0 になり勾配が伝わらない（**飽和**）、小さすぎると層を通るたびに信号と勾配が 0 へ縮んで消える（**勾配消失**）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`tanh` のグラフの左右の端（入力が非常に大きい場所）の傾きと、小さな数（0.01）を何度も掛け算したときの出力の大きさを想像しましょう。
- **ヒント 2**：重み 1.0 では出力が ±1 に張り付いて傾きほぼ 0（飽和）、重み 0.01 では数層で出力が 0.0 に潰れます（勾配消失）。
- **正解**：`d`

**考え方**：深いネットワークで信号も勾配も潰さず奥まで届けるには、各層を通っても標準偏差が極端に膨らんだり縮んだりしない初期スケールが不可欠です。

1. 重み 1.0（大きすぎ）：256個の和で分散が256倍 → `tanh` が ±1 に張り付き勾配ほぼ 0（飽和）。
1. 重み 0.01（小さすぎ）：毎回縮んで4〜5層目で `0.0`（勾配消失）。
1. 重み $1 / \sqrt{256}$：256個足して膨らむ $\sqrt{256} = 16$ 倍をちょうど打ち消し、深い層まで信号が届く。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/10-training-tips.html#ex-ch10-q05

</details>

## 治療1：LayerNorm（層正規化）

初期化に気をつけても、学習中にパラメータが変わると、また数の大きさは暴れ出します。そこで、**毎回、強制的に大きさをそろえる**部品を挟みます。それが **LayerNorm**（レイヤーノーム、層正規化）です。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
h = torch.randn(2, 6) * 5 + 3      # 平均3、標準偏差5くらいの、暴れた数
mean = h.mean(dim=1, keepdim=True)
var = h.var(dim=1, keepdim=True, unbiased=False)   # unbiased=False：数式どおり N で割る
h_hand = (h - mean) / torch.sqrt(var + 1e-5)       # 分散に ε=1e-5 を足してから平方根をとる

ln = nn.LayerNorm(6, elementwise_affine=False)
print(h_hand)
print(ln(h))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
tensor([[-0.2989,  0.0184, -1.2615,  1.8086,  0.5781, -0.8448],
        [-0.0293,  0.7320,  1.0004, -0.1376, -2.0512,  0.4857]])
tensor([[-0.2989,  0.0184, -1.2615,  1.8086,  0.5781, -0.8448],
        [-0.0293,  0.7320,  1.0004, -0.1376, -2.0512,  0.4857]])
```

</details>

#### 📐 数学のポイント：LayerNorm（層正規化）：9ステップで理由から理解する

**① 解決したい問題**：学習の途中で層の出力が急に大きくなったり小さくなったりしても、毎ステップ安定して「ちょうどいい大きさ（平均0・標準偏差1付近）」に保ちたい。

**② 素朴な方法の落とし穴**：
- 単純に $(x_i - \mu) / \sigma$ とすると、もし全要素が同じ値（例：`[5, 5, 5]`）のとき分散 $\sigma^2 = 0$ になり、**ゼロ除算で `nan` が発生**して学習が一撃で崩壊します。
- また「強制的に平均0・標準偏差1」に縛りすぎると、次の層が欲しい範囲（たとえば少し大きめ／少し右寄りの値）を表現しにくくなります。

**③ 完成した数式**：
$$\text{LayerNorm}(x)_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma_i + \beta_i$$

**④ 記号の読み方テーブル**：

| 記号 | 読み方 | 役割 |
|---|---|---|
| $\mu$ | ミュー | その1データ（1行）の特徴ベクトル全体の平均 |
| $\sigma^2$ | シグマ2乗 | その1データ（1行）の母分散（$N$ で割る、`unbiased=False`） |
| $\epsilon$ | イプシロン | ゼロ除算防止のための極小の正の定数（通常 $10^{-5}$ = `1e-5`） |
| $\gamma_i$ | ガンマ | 正規化後に掛ける**学習可能な倍率**（初期値 `1`）。必要なスケールを回復できる！ |
| $\beta_i$ | ベータ | 正規化後に足す**学習可能なシフト**（初期値 `0`）。必要な中心位置を回復できる！ |

**⑤ 小さな3数で手計算**：`x = [2, 4, 6]`（$\gamma=1, \beta=0, \epsilon=10^{-5}$）のとき：
1. 平均 $\mu = (2 + 4 + 6) / 3 = 4$
2. 分散 $\sigma^2 = ((-2)^2 + 0^2 + 2^2) / 3 = 8 / 3 \approx 2.6667$
3. 分母 $\sqrt{\sigma^2 + 10^{-5}} \approx \sqrt{2.6667} \approx 1.6330$
4. 正規化後：
   - $(2 - 4) / 1.6330 \approx -1.225$
   - $(4 - 4) / 1.6330 = 0.000$
   - $(6 - 4) / 1.6330 \approx +1.225$

元の数が `[200, 400, 600]`（100倍）でも `[102, 104, 106]`（+100）でも、結果はまったく同じ `[-1.225, 0.0, +1.225]` にピタッとそろいます！

```python no-run
import math
x = [2.0, 4.0, 6.0]
mu = sum(x) / len(x)
var = sum((v - mu) ** 2 for v in x) / len(x)
normed = [(v - mu) / math.sqrt(var + 1e-5) for v in x]
print([round(v, 3) for v in normed])   # → [-1.225, 0.0, 1.225]
```

> **💡 理論のポイント：LayerNormは「偏差値」**
>
> LayerNormは、1つのデータ（1行、つまり1つの文字やトークンの特徴量ベクトル）の中の数について、**平均を引いて標準偏差で割る**だけです。すると、元の数がどんなに大きくても小さくても、平均0・標準偏差1にそろいます。
>
> これは、テストの点数を **偏差値**（Zスコア）にするのと同じ発想です。難しいテストでも簡単なテストでも、偏差値に変換すれば同じものさしで比べられますよね。画像認識でよく使われるバッチ正規化と違って、他のデータ（バッチの仲間）に一切依存せず、そのデータ単独で完結して計算できるため、文章を1文字ずつ扱う自然言語処理やLLMと非常に相性が良いのが特長です。
>
> 実際の `nn.LayerNorm` は、そろえた後に「学習可能な倍率 $\gamma$ と足し算 $\beta$」で少しだけ形を調整できるようになっています（上のコードでは `elementwise_affine=False` でそれを切っています）。LayerNormはGPTの部品として、第12章で何度も登場します。

#### 📝 確認問題（`ch10-q06` · Lv.2 · LayerNorm（γ=1, β=0）の具体的な手計算）

5要素のベクトル `x = [2, 4, 5, 6, 8]` に倍率・足し算なし（$\gamma=1, \beta=0$）の LayerNorm をかけたとき、5番目の要素 `8` はいくつに変換されますか？（$\epsilon$ は極めて小さいものとして無視、誤差 $\pm 0.02$ まで正解）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：LayerNorm（$\gamma=1, \beta=0$）は「各要素から平均 $\mu$ を引き、母標準偏差 $\sigma$ で割る」変換です。まず平均と母分散を求めましょう。
- **ヒント 2**：平均 $\mu = 5$、ずれは `[-3, -1, 0, +1, +3]`、母分散 $\sigma^2 = (9+1+0+1+9)/5 = 4$ なので標準偏差は $\sigma = 2$ です。$(8 - 5) / 2$ を計算しましょう。
- **正解**：`1.5`

**考え方**：LayerNorm は $(x_i - \mu) / \sqrt{\sigma^2 + \epsilon}$ なので、`[2, 4, 5, 6, 8]` は平均 5・標準偏差 2 となり、`[-1.5, -0.5, 0.0, 0.5, 1.5]` へ変換されます。

1. 平均：$\mu = (2 + 4 + 5 + 6 + 8) / 5 = 25 / 5 = 5$。
1. ずれの2乗：$(-3)^2 + (-1)^2 + 0^2 + 1^2 + 3^2 = 9 + 1 + 0 + 1 + 9 = 20$。
1. 母分散：$\sigma^2 = 20 / 5 = 4$ → 標準偏差 $\sigma = \sqrt{4} = 2$。
1. 5番目の要素：$(8 - 5) / 2 = 1.5$。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/10-training-tips.html#ex-ch10-q06

</details>

#### 📝 確認問題（`ch10-q07` · Lv.2 · LayerNorm の数式における ε・γ・β の役割）

LayerNorm の式 $\text{LayerNorm}(x)_i = \dfrac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma_i + \beta_i$ において、小さな定数 $\epsilon$（イプシロン、例：`1e-5`）と、学習可能パラメータ $\gamma$（ガンマ）・$\beta$（ベータ）の役割として**正しい**組み合わせはどれですか？

- **a.** $\epsilon$ は分散 $\sigma^2 = 0$ のときのゼロ除算（`nan`）を防ぐ安全装置、$\gamma$（倍率）と $\beta$（シフト）は正規化で縛られすぎないよう最適なスケールと中心を学習で復元するため
- **b.** $\epsilon$ はミニバッチ内の他サンプルの平均と混ぜ合わせるための重みであり、$\gamma$ と $\beta$ は出力層の語彙数 $V$ に合わせて次元数を拡大するための変換行列である
- **c.** $\epsilon$ はオプティマイザの学習率を調整する係数であり、$\gamma$ と $\beta$ は過学習を防ぐために毎ステップランダムにニューロンを 0 に落とすための乱数マスクである
- **d.** $\epsilon$ は正規化後の平均を強制的に 1 へ引き上げる定数であり、$\gamma$ と $\beta$ は重みの発散を防ぐため常に 0 に固定して学習では一切更新しないパラメータである

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：入力ベクトルが `[5, 5, 5]` のように全部同じ値だったとき分母はどうなるでしょう？ また「必ず平均 0・標準偏差 1」に固定されすぎると表現力はどうなるでしょう？
- **ヒント 2**：$\epsilon$ は分母のゼロ除算を防ぎ、学習可能な倍率 $\gamma$（初期値 1）とシフト $\beta$（初期値 0）が必要に応じたスケールと中心位置を回復させます。
- **正解**：`a`

**考え方**：$\epsilon$ が数値安定性（ゼロ除算防止）を守り、学習可能な $\gamma, \beta$ が表現力（必要なら元のスケールやオフセットに戻せる自由度）を守ります。

1. $\epsilon$（例：`1e-5`）：全要素が同じ値で $\sigma^2 = 0$ になっても、分母 $\sqrt{0 + 10^{-5}} > 0$ で `nan` を防ぐ。
1. $\gamma$（初期値 `1`）：正規化後の倍率を次元ごとに学習。
1. $\beta$（初期値 `0`）：正規化後の足し算（中心位置）を次元ごとに学習。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/10-training-tips.html#ex-ch10-q07

</details>

*(この節の追加問題 `ch10-q08` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/10-training-tips.html で自動採点・復習できます)*

## 治療2：Adam（賢い坂の下り方）

第7章からずっと、坂の下り方は「傾きに学習率を掛けて進む」SGDでした。現代のLLMの学習では、ほぼ必ず **Adam**（アダム）またはその改良版 **AdamW** を使います。

> **💡 理論のポイント：Adamは「慣性」と「自分専用の歩幅」**
>
> SGDの弱点は2つあります。
>
> 1. **ミニバッチのぶれに振り回される**：毎回の勾配がガタガタなので、ジグザグに進んでしまう
> 2. **全パラメータが同じ歩幅**：めったに出てこない文字の埋め込みは、勾配がたまにしか来ないので、なかなか学習が進まない
>
> Adamは、歴史的な2つの名アイデア（**モメンタム** と **RMSprop**）を組み合わせることで、これを一挙に解決します。
>
> - **慣性（モーメンタム / 勾配の向きの記憶）**：これまでの勾配の平均的な向きを覚えておき、それに沿って進む。重いボールが坂を勢いよく転がるとき、小さなでこぼこがあっても直進し続けるのと同じです。
> - **パラメータごとの歩幅（RMSprop / 勾配の2乗の記憶）**：勾配がいつも小さいパラメータは「まだ学び足りない」と判断して歩幅を大きく、いつも勾配が大きいパラメータは「慎重に進もう」と歩幅を小さくします。各パラメータの勾配の2乗平均の平方根（RMS）で割ることで、それぞれにぴったりの歩幅を自動調整します。
>
> AdamWの「W」は **重み減衰**（Weight Decay）を正しく組み込んだ版、という意味です。従来のAdamでは重み減衰を勾配と混ぜて計算していたため、歩幅の自動調整のせいで重み減衰が歪んでしまう問題がありました。AdamWは歩幅調整とは独立してきれいに重みを減衰させるよう修正したもので、現代のLLM学習のデファクトスタンダード（標準）となっています。

<details><summary><b>基礎の補足：AdamW の中身を3行の直感でつかむ</b>（クリックで開く）</summary>

毎ステップの勾配を $g$ とすると、AdamW は各パラメータごとに次の2つの移動平均を記憶して更新します（$\beta_1 \approx 0.9, \; \beta_2 \approx 0.999$）：

| 部品 | 直感的な式 | 役割 |
|---|---|---|
| ① 慣性（モメンタム）$m$ | $m \leftarrow 0.9 m + 0.1 g$ | 過去の勾配の向きを覚えてジグザグを打ち消す |
| ② 勾配の大きさの記憶 $v$ | $v \leftarrow 0.999 v + 0.001 g^2$ | いつも勾配が大きいパラメータほど $\sqrt{v}$ が大きくなる |
| ③ 更新＋独立した重み減衰 | $w \leftarrow (1 - \eta \lambda) w - \eta \dfrac{m}{\sqrt{v} + \epsilon}$ | 歩幅を $\sqrt{v}$ で自動調整しつつ、独立して重みを少し縮める（これが **W**！） |

※ 実際には初期ステップの偏りを直す補正（bias correction）も入りますが、本質はこの3つです！

</details>

では、AdamWで学習させてみましょう。学習の様子を記録しながら、6000ステップ進めます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def train(model, steps=6000, lr=1e-3, weight_decay=0.0, schedule=False, seed=0):
    g = torch.Generator().manual_seed(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = []
    for step in range(steps + 1):
        if schedule:                                   # 学習率スケジュール（後で説明）
            for group in opt.param_groups:
                group["lr"] = lr_at(step, lr, steps)
        ix = torch.randint(0, len(train_data) - T, (256,), generator=g)
        x = torch.stack([train_data[i:i + T] for i in ix])
        y = train_data[ix + T]
        loss = F.cross_entropy(model(x), y)
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # 勾配クリッピング（後で説明）
        opt.step()
        if step % 1000 == 0:
            history.append((step, loss.item(), val_loss(model)))
            print(f"ステップ {step:4d}：訓練 {history[-1][1]:.3f}  検証 {history[-1][2]:.3f}")
    return history

torch.manual_seed(0)
model_a = MLP(V, T)
hist_a = train(model_a, lr=1e-3)

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
ステップ    0：訓練 7.678  検証 7.660
ステップ 1000：訓練 3.540  検証 3.581
ステップ 2000：訓練 2.765  検証 3.419
ステップ 3000：訓練 2.775  検証 3.386
ステップ 4000：訓練 2.434  検証 3.392
ステップ 5000：訓練 2.393  検証 3.428
ステップ 6000：訓練 1.937  検証 3.465
```

</details>

速い！ SGDでは1万ステップかかった3.4台に、2000ステップで到達しました。でも、よく見てください。3000ステップを過ぎると、**訓練損失は下がり続けているのに、検証損失は上がり始めて**います。

#### 📝 確認問題（`ch10-q09` · Lv.2 · AdamW の2大機能（慣性・パラメータごとの歩幅）と W（重み減衰分離））

Adam の「**慣性（モーメンタム）**」は、SGD のどのような問題を和らげますか？

- **a.** ミニバッチごとに勾配の向きがぶれてジグザグに進んでしまう問題を、過去の勾配の平均的な向きを覚えることで滑らかに直進させる
- **b.** 過去の勾配の勢いだけで次の更新方向を完全に予測できるため、誤差逆伝播法（`loss.backward()`）の呼び出し自体を不要にする
- **c.** 学習の途中で重要度の低い重みを自動的に削除することで、モデルのパラメータ総数を半分近くに削減してメモリ消費を抑える
- **d.** 過去のミニバッチの文章を自動で組み合わせて新しい訓練サンプルを生成し、実質的な訓練データ量を 2 倍以上に水増しする

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：重い鉄球がデコボコの坂を転がり下りるとき、小さな小石（ミニバッチのノイズ）に当たっても進路はどう保たれるでしょう？
- **ヒント 2**：過去の勾配の移動平均（慣性）を保つことで、ミニバッチごとのジグザグなぶれを打ち消し、平均的な下り方向へ滑らかに進めます。
- **正解**：`a`

**考え方**：元の章末クイズの復習です。Adam = モーメンタム（慣性）＋ RMSprop（パラメータごとの歩幅調整）のハイブリッドです。

1. 慣性（モーメンタム）：過去の勾配の移動平均 $m$ で向きを安定化。
1. 適応歩幅（RMSprop）：過去の勾配の2乗の移動平均 $v$ の平方根で割り、よく動くパラメータは慎重に・あまり動かないパラメータは大きめに更新。
1. 結果として SGD の 1/5 のステップ数で同じ検証損失に到達するほど速く賢く下れる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/10-training-tips.html#ex-ch10-q09

</details>

## 診察3：過学習

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def plot_history(hists, labels, title):
    plt.figure(figsize=(6, 3.5))
    for h, label in zip(hists, labels):
        s = [r[0] for r in h]
        plt.plot(s, [r[1] for r in h], linestyle="--", label=label + "（訓練）")
        plt.plot(s, [r[2] for r in h], label=label + "（検証）")
    plt.ylim(1.5, 4.5)
    plt.xlabel("ステップ"); plt.ylabel("損失")
    plt.legend(); plt.title(title)
    plt.show()

plot_history([hist_a], ["AdamW"], "訓練は下がり続け、検証は途中から悪化")

![AdamWで学習したMLPの学習曲線。訓練損失は下がり続けるが、検証損失は3000ステップ付近を底に上がっていく](https://sakunokyumei.github.io/llm-research-textbook/img/figs/10-training-tips/cell8_0.png)

> **💡 理論のポイント：過学習（オーバーフィッティング）**
>
> 訓練データに合わせすぎて、検証データでの成績が悪くなる現象を **過学習**（overfitting）と言います。過去問の答えを丸暗記して、本番で応用がきかない状態です。
>
> 過学習が起きやすいのは、
>
> - データが少ない（『こころ』の16万字は、LLMの世界では米粒ほど）
> - モデルが大きい（パラメータが多いほど、丸暗記の容量がある）
> - 長く学習しすぎる
>
> ときです。治療法は「データを増やす」が一番ですが（第13章でやります）、ここでは**モデルの側で丸暗記しにくくする**方法を2つ使います。

#### 📝 確認問題（`ch10-q10` · Lv.2 · 学習曲線からの過学習診断と治療法の選択）

学習中、訓練損失は `2.0` まで下がり続けていますが、検証損失は `3.5` を底にして `3.7` へ上がり始めました。何が起きており、有効な対策はどれですか？

- **a.** **勾配消失**が起きている。深い層まで信号が届いていないため、訓練データをさらに減らしてバッチサイズを極端に大きくするのが有効
- **b.** **過学習**が起きている。データを増やす、ドロップアウトや重み減衰を強める、モデルを小さくする、検証損失が最小だった時点で止める、などが有効
- **c.** **学習不足**が起きている。モデル容量が足りないためさらに長く学習を続け、層数や隠れ層を 10 倍に増やしてドロップアウトを `0` にするのが有効
- **d.** **初期化ミス**が起きている。最初の出力のばらつきが小さすぎるため、出力層の重みを 30 倍に拡大して最初からやり直すのが有効

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：訓練損失は順調に下がっているのに、初見の検証損失だけが底を打って U 字に上がり始めている状態が何を意味するか考えましょう。
- **ヒント 2**：訓練データの丸暗記（過学習）です。正則化（ドロップアウト・重み減衰）、データ増量、早期終了（Early Stopping）などが有効な治療法になります。
- **正解**：`b`

**考え方**：元の章末クイズの確認です。本文の AdamW 治療前（`hist_a`）でも3000ステップを境にまさに同じU字の悪化が起きました。

1. 症状：訓練損失 ↓（2.0）、検証損失 ↑（3.5 → 3.7）＝ 過学習。
1. 原因：データ量に対してモデルの記憶容量が大きすぎる／学習が長すぎる。
1. 治療：データ増量・ドロップアウト・重み減衰・早期終了（Early Stopping）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/10-training-tips.html#ex-ch10-q10

</details>

## 治療3：ドロップアウトと重み減衰

> **💡 理論のポイント：ドロップアウト：わざと一部を休ませる**
>
> **ドロップアウト**（dropout）は、学習中に、ニューロンの一部（例えば30%）を**ランダムに0にする**方法です。毎回違うニューロンが休むので、モデルは「特定のニューロンに頼った丸暗記」ができなくなり、どのニューロンが休んでもなんとかなるような、頑丈な特徴を学ぶようになります。
>
> チームで練習するとき、毎回ランダムに3割が欠席するとしたら、「エースに全部任せる」作戦は使えませんよね。全員がそこそこできるようになるしかない。それがドロップアウトです。
>
> 評価（検証や生成）のときは、全員出席（ドロップアウトなし）にします。そのために `model.eval()`（評価モード）と `model.train()`（学習モード）を切り替えます。準備セルの `val_loss` 関数の中で、これをやっていました。

<details><summary><b>基礎の補足：`model.train()` と `model.eval()` の違い、および重み減衰の仕組み</b>（クリックで開く）</summary>

| モード | ドロップアウトの挙動 | 使う場面 |
|---|---|---|
| `model.train()` | 確率 $p$（例：`0.3`）でランダムに `0` にし、残りの要素を $\frac{1}{1-p}$ 倍して平均の大きさを保つ | 訓練ループの中 |
| `model.eval()` | ドロップアウトを完全にオフ（入力をそのまま通す＝毎回同じ決定的な出力） | 検証 `val_loss`・文章生成 `generate` |

**重要**：`model.eval()` は層の挙動切替、`with torch.no_grad():` は勾配記録の停止という**別の機能**なので、評価時は必ずセットで使います（評価が終わったら `model.train()` に戻すのも忘れずに！）。

また **重み減衰（Weight Decay）** は、毎ステップ重みを $w \leftarrow (1 - \eta \lambda) w$ のようにほんの少し 0 方向へ縮める正則化です。大きな重みほど強く引き戻されるため、特定の1文だけに極端に反応する丸暗記を防げます。

</details>

> **💡 理論のポイント：重み減衰：大きな重みに罰金**
>
> **重み減衰**（weight decay）は、毎ステップ、重みを少しだけ0に向かって縮める方法です。極端に大きな重みを持つ（＝特定のパターンに極端に反応する）ことを防ぎます。第8章で話した **正則化** の代表選手で、第5章のスムージングの親戚です。AdamWでは `weight_decay=0.1` のように指定します。

#### 📝 確認問題（`ch10-q11` · Lv.2 · ドロップアウトの model.train() / model.eval() 切替と重み減衰）

ドロップアウト（`nn.Dropout(0.3)`）を入れたモデルを検証・生成するとき、`model.eval()` を呼び忘れて学習モード（`model.train()`）のまま評価してしまうと何が起きますか？

- **a.** `model.train()` のまま検証データを流すと内部で自動的に逆伝播が走り、検証データの答えに合わせてモデルの重みが勝手に書き換わってしまう
- **b.** 評価の結果には何も影響しない。`model.eval()` はコードの読み手に「ここは評価箇所だ」と伝えるためのコメントと同じ飾りである
- **c.** `model.eval()` を呼び忘れると直前に書いた `with torch.no_grad():` の指定まで自動的に無効化され、GPU メモリ消費が数倍に膨れ上がってしまう
- **d.** 評価時にも毎回ランダムに 30% のニューロンが 0 に休まされてしまい、同じモデル・同じデータなのに実行するたびに検証損失がぶれて本来の性能より悪く出る

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：ドロップアウトは学習中（`model.train()`）と本番評価中（`model.eval()`）で、ニューロンを休ませるかどうかどう切り替わるでしょうか？
- **ヒント 2**：`model.train()` のままだと検証時にも毎回ランダムに 30% のユニットが 0 になるため、同じデータでも結果がぶれて本来の実力より悪い値になります。
- **正解**：`d`

**考え方**：評価関数 `val_loss` の先頭で `model.eval()`、末尾で `model.train()` に戻すセットの書き方は、PyTorch実務の必須定型文です。

1. 学習時（`model.train()`）：ランダムに30%を0にし、残りを `1 / 0.7` 倍して期待値を保つ。
1. 評価時（`model.eval()`）：ドロップアウトをオフにして全ニューロンを使い、決定的に評価する。
1. 評価後に学習を再開するなら、忘れずに `model.train()` へ戻す。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/10-training-tips.html#ex-ch10-q11

</details>

## 治療4：学習率スケジュール

最後に、学習率を学習の途中で変える技です。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def lr_at(step, max_lr, total, warmup=200):
    if step < warmup:
        return max_lr * step / warmup                       # ウォームアップ：0から徐々に上げる
    progress = (step - warmup) / (total - warmup)
    return max_lr * 0.5 * (1 + math.cos(math.pi * progress))  # コサイン減衰：なめらかに0へ

steps = list(range(6001))
plt.figure(figsize=(6, 2.8))
plt.plot(steps, [lr_at(s, 2e-3, 6000) for s in steps])
plt.xlabel("ステップ"); plt.ylabel("学習率")
plt.title("ウォームアップ＋コサイン減衰")
plt.show()

![学習率スケジュールのグラフ。最初の200ステップで0から最大値まで上がり、その後コサインカーブでなめらかに0まで下がる](https://sakunokyumei.github.io/llm-research-textbook/img/figs/10-training-tips/cell9_0.png)

> **💡 理論のポイント：なぜ学習率を変えるのか**
>
> - **最初（ウォームアップ）**：学習の最初は、パラメータがでたらめで勾配も荒れています。いきなり大股で進むと転ぶので、小股から始めて徐々に歩幅を広げます
> - **最後（減衰）**：谷底に近づいたら、大股だと谷底を行ったり来たりしてしまいます（第7章の学習率が大きすぎる例）。歩幅を小さくして、そっと谷底に着地します
>
> **コサイン減衰**は、三角関数のコサインのカーブを使って、なめらかに学習率を下げる方法です。GPTを含む多くのLLMで使われています。

> **🐍 Python・PyTorchのポイント：opt.param_groups**
>
> `train` 関数の中の `for group in opt.param_groups: group["lr"] = …` が、学習率を変えている部分です。`opt.param_groups` は、オプティマイザーが持っている設定（学習率など）の入れ物のリストで、`group["lr"]` を書き換えると、次の `opt.step()` からその学習率で進みます。`schedule=True` のときは、これを毎ステップ `lr_at` の値に書き換えています。

> **💡 理論のポイント：勾配クリッピング**
>
> `train` 関数の中の `clip_grad_norm_(model.parameters(), 1.0)` は、**勾配の大きさが1.0を超えたら、1.0に縮める**処理です。たまに来る異常に大きな勾配（勾配爆発）で、一気に遠くへ吹き飛ばされるのを防ぐ安全ベルトです。向きはそのままで、長さだけを制限します。

#### 📝 確認問題（`ch10-q12` · Lv.2 · ウォームアップ＋コサイン減衰 lr_at の値の変化予測）

本文の学習率スケジュール関数 `lr_at` のウォームアップ期間と終了時の値を確認します。次のコードを実行すると何が表示されますか？

```python
import math

def lr_at(step, max_lr, total, warmup=200):
    if step < warmup:
        return max_lr * step / warmup
    progress = (step - warmup) / (total - warmup)
    return max_lr * 0.5 * (1 + math.cos(math.pi * progress))

print(round(lr_at(100, 2.0, 1000, warmup=200), 2), round(lr_at(200, 2.0, 1000, warmup=200), 2))
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`step < warmup` の間は `max_lr * step / warmup` で直線的に増え、`step == warmup` では `progress = 0` になります。
- **ヒント 2**：`step = 100` では `2.0 * 100 / 200`、`step = 200` では `cos(0) = 1` を使って `2.0 * 0.5 * (1 + 1)` をそれぞれ計算しましょう。
- **期待される出力**：`1.0 2.0`

**考え方**：ウォームアップで `0` から `max_lr` まで直線的に立ち上がり、`step == warmup` でちょうど最大値 `max_lr` に到達します。

1. `step = 100 < 200`：`2.0 * 100 / 200 = 1.0`。
1. `step = 200`：`progress = 0.0` → `2.0 * 0.5 * (1 + cos(0)) = 2.0 * 0.5 * 2 = 2.0`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/10-training-tips.html#ex-ch10-q12

</details>

#### 📝 確認問題（`ch10-q13` · Lv.2 · ウォームアップ・コサイン減衰・勾配クリッピングの役割）

学習率スケジュール（ウォームアップ＋コサイン減衰）と、勾配クリッピング `clip_grad_norm_(model.parameters(), 1.0)` それぞれの役割の組み合わせとして**正しい**のはどれですか？

- **a.** ウォームアップは初期の荒れた勾配での転倒を防ぎ、コサイン減衰は終盤に小股で谷底へ着地させ、勾配クリッピングは異常に巨大な勾配を向きを保ったまま上限 1.0 に縮めて爆発を防ぐ
- **b.** AdamW は内部で適応的に歩幅を自動制御しているため、ウォームアップや勾配クリッピングを併用すると更新が競合してエラーになり学習が止まる
- **c.** ウォームアップは学習の最終盤に向けて学習率を段階的に最大値まで引き上げる処理であり、コサイン減衰は最初の数ステップだけ急激に学習率を下げる処理である
- **d.** 勾配クリッピングは全パラメータの中で勾配が 1.0 を超えた要素を異常値と見なして削除し、その重みを強制的に 0 にリセットする処理である

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：「学習の最初に歩幅をどうするか」「終盤にどう着地するか」「突然の巨大な勾配が来たときの安全ベルトは何か」の 3 点で整理しましょう。
- **ヒント 2**：ウォームアップ＝初期の小股スタート、コサイン減衰＝終盤の滑らかな減速、勾配クリッピング＝勾配の向きを保ったままノルム上限を 1.0 に抑える安全装置です。
- **正解**：`a`

**考え方**：これら3つの安定化技を揃えることで、大きめの最大学習率（`2e-3` など）でも吹き飛ばずに速く深く下れるようになります。

1. ウォームアップ：初期化直後の荒れた勾配で暴走しないよう小股から開始。
1. コサイン減衰：終盤に谷底を飛び越えないよう歩幅を滑らかに縮小。
1. 勾配クリッピング：全勾配のノルムが `max_norm=1.0` を超えたら比率縮小（安全ベルト）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/10-training-tips.html#ex-ch10-q13

</details>

## 全部の治療を施す

ドロップアウト30%、重み減衰0.1、学習率スケジュール、勾配クリッピング。全部入りで学習させてみましょう。

🔎 **実行前に確認**：コードの各行が何を受け取り、何を作るか確認してから ▶ を押してください。

In [ ]:
torch.manual_seed(0)
model_b = MLP(V, T, dropout=0.3)
hist_b = train(model_b, lr=2e-3, weight_decay=0.1, schedule=True)

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
ステップ    0：訓練 7.706  検証 7.703
ステップ 1000：訓練 3.533  検証 3.530
ステップ 2000：訓練 2.856  検証 3.347
ステップ 3000：訓練 2.917  検証 3.272
ステップ 4000：訓練 2.656  検証 3.213
ステップ 5000：訓練 2.831  検証 3.183
ステップ 6000：訓練 2.469  検証 3.177
```

</details>

🔎 **実行前に確認**：コードの各行が何を受け取り、何を作るか確認してから ▶ を押してください。

In [ ]:
plot_history([hist_a, hist_b], ["治療前", "治療後"], "治療前と治療後")

![治療前と治療後の学習曲線の比較。治療後は検証損失が最後まで下がり続け、3.18付近に到達する](https://sakunokyumei.github.io/llm-research-textbook/img/figs/10-training-tips/cell11_0.png)

検証損失が最後まで下がり続け、**3.177** に到達しました。訓練損失と検証損失の差も小さくなっています。治療成功です。

| モデル | 章 | 検証データの損失 |
|---|---|---|
| でたらめ | 5 | 7.631 |
| bigram（数える） | 5 | 3.581 |
| ニューラルbigram | 8 | 3.685 |
| MLP（SGD） | 9 | 3.395 |
| **MLP（LayerNorm＋AdamW＋ドロップアウト＋スケジュール）** | **10** | **3.177** |

## 学習したモデルを保存する

2分かけて学習したモデルも、Colabの接続が切れたら消えてしまいます。**Googleドライブ**に保存しておきましょう。

🔎 **実行前に確認**：コードの各行が何を受け取り、何を作るか確認してから ▶ を押してください。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

実行すると、Googleアカウントへのアクセスを許可するか聞かれます。自分のアカウントを選んで許可してください。すると `/content/drive/MyDrive` が、あなたのGoogleドライブの中身になります。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import os
save_dir = "/content/drive/MyDrive/tsukuru-llm"
os.makedirs(save_dir, exist_ok=True)
torch.save(model_b.state_dict(), f"{save_dir}/mlp_kokoro.pt")
print("保存しました：", os.listdir(save_dir))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
保存しました： ['mlp_kokoro.pt']
```

</details>

読み込むときは、同じ設計図でモデルを作ってから、保存した中身を流し込みます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
loaded = MLP(V, T, dropout=0.3)
loaded.load_state_dict(torch.load(f"{save_dir}/mlp_kokoro.pt", weights_only=True))
print("読み込んだモデルの検証損失：", round(val_loss(loaded), 3))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
読み込んだモデルの検証損失： 3.177
```

</details>

> **🐍 Python・PyTorchのポイント：state_dict**
>
> `model.state_dict()` は、モデルの全パラメータを「名前 → テンソル」の辞書にしたものです。保存されるのは**中身の数だけ**で、設計図（クラスの定義）は保存されません。だから読み込むときは、先に同じクラスでモデルを作る必要があります。ゲームのセーブデータを読み込むには、ゲーム本体が必要なのと同じです。

> **¥0 無料枠のコツ：ドライブの容量**
>
> 無料のGoogleアカウントのドライブ容量は15GB（Gmailや写真と共有）です。このMLPの保存ファイルは約2.6MBなので心配いりませんが、第17章以降の大きなモデル（数百MB〜数GB）を保存するときは、容量に気をつけましょう。いらなくなったファイルは消すのが0円のコツです。

#### 📝 確認問題（`ch10-q14` · Lv.2 · state_dict による保存・読み込みの仕組みと注意点）

`torch.save(model.state_dict(), path)` で保存したファイルを後から読み込むとき、`loaded = MLP(V, T, ...)` と**同じ設計図（同じ `d, H, T, V`）でモデルを作り直してから `loaded.load_state_dict(...)` を呼ぶ必要がある**理由はどれですか？

- **a.** `dropout` の確率設定だけは保存されないため、`MLP(...)` を `dropout=0.0` で作り直してから読み込まないと必ず形状不一致エラーが発生するから
- **b.** `state_dict()` に保存されるのはパラメータの「名前 → テンソル（数値の中身）」の辞書だけであり、クラスの設計図や構造自体は保存されないから
- **c.** `state_dict()` に保存されるのは学習中の訓練損失と検証損失の履歴リストだけであり、モデルの重み本体は別のファイルへ自動保存される仕様だから
- **d.** 保存したファイルを読み込んだ直後は必ずモデルを最初から再学習させる仕様になっており、空のモデルを作り直さないとオプティマイザが初期化できないから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`model.state_dict()` の中身を `print` したとき、Python の `class MLP` のコード自体が入っているか、それとも数値の辞書だけか思い出しましょう。
- **ヒント 2**：ゲームのセーブデータ（数値の中身）を読むには同じゲーム本体（同じ `d, H, T, V` で作った `MLP`）が必要なように、同じ構造の入れ物を作ってから流し込みます。
- **正解**：`b`

**考え方**：`state_dict()` 形式で保存すると、Python のクラス定義に依存せず軽量かつ安全に重みを受け渡しできるため、PyTorch 公式でも推奨されています。

1. 保存：`torch.save(model_b.state_dict(), "mlp_kokoro.pt")`（中身の数値辞書だけ保存）。
1. 復元①：`loaded = MLP(V, T, dropout=0.3)`（同じ構造の空の入れ物を用意）。
1. 復元②：`loaded.load_state_dict(torch.load("mlp_kokoro.pt"))`（数値を流し込む）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/10-training-tips.html#ex-ch10-q14

</details>

## 学習の診察表

この章のまとめとして、症状と治療法の表を作っておきます。学習がうまくいかないときは、上から順に確認しましょう。

| 症状 | 考えられる原因 | 治療法 |
|---|---|---|
| 最初の損失が $\log V$ よりずっと大きい | 初期化が大きすぎる | 出力層の重みを小さく。デフォルトの初期化を使う |
| 損失がまったく下がらない | 学習率が小さすぎる／バグ（`zero_grad` 忘れ、`requires_grad` 忘れ） | 学習率を10倍に。コードを見直す |
| 損失が `nan` になる、急に跳ね上がる | 学習率が大きすぎる／勾配爆発 | 学習率を下げる。勾配クリッピング。ウォームアップ |
| 損失がガタガタで下がりにくい | バッチが小さい／学習率が大きい | バッチを大きく。学習率スケジュール |
| 訓練だけ下がって、検証が上がる | 過学習 | データを増やす。ドロップアウト。重み減衰。早めに止める |
| 訓練も検証も高いまま横ばい | モデルが小さすぎる（表現力不足） | 層や幅を増やす。窓（文脈）を広げる |

> **✨ コツ：研究者の口ぐせ：「まず小さなデータで過学習させてみよ」**
>
> 新しいモデルを作ったら、最初に**ごく少ないデータ（例えば1バッチだけ）で学習**させて、訓練損失がほぼ0まで下がるか確かめます。下がらなければ、モデルか学習ループにバグがあります。丸暗記すらできないモデルが、汎化できるはずがないからです。

> **⚠️ つまずき注意：よくあるつまずき**
>
> - **`drive.mount` で止まる・失敗する**：ポップアップがブロックされていないか確認しましょう。アカウント選択の画面で、すべての項目にチェックを入れて「続行」を押します。
> - **`RuntimeError: Error(s) in loading state_dict`**：保存したときと違う設計図（`dropout` 以外の `d` や `H`、`T` など）でモデルを作っています。保存時と同じ設定で作りましょう。
> - **検証損失が、同じモデルなのに実行するたびにぶれる**：`model.eval()` と `model.train()` の切り替えを忘れていないか確認しましょう。評価時にドロップアウトが効いていると、結果がぶれます。
> - **`.std()` で計算した標準偏差が、手計算と合わない**：PyTorchの `.std()` と `.var()` は、標準では N ではなく N−1 で割ります（統計で「標本から全体を推定する」ときの割り方）。数式どおり N で割るには `unbiased=False` を付けます。LayerNormは N で割るほうです。

#### 📝 確認問題（`ch10-q15` · Lv.3 · 学習トラブルの診察手順と「まず1バッチで過学習させる」検証法）

新しいニューラルモデルを学習させ始めたとき、①「損失がまったく下がらない」、②「途中で損失が `nan` になった」という2つの症状が出たら、それぞれ何を疑ってどう処置するか説明してください。また、本番学習の前に「まず1バッチだけで過学習させてみる」のはなぜ有効かも説明してください。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：「下がらない」「`nan` になる」のそれぞれで最初に疑うべき学習率・コードバグ・安定化策を思い出し、本番前の 1 バッチ過学習テストの役割と結びつけましょう。
- **ヒント 2**：①下がらない＝lr過小や `zero_grad`/`backward` 周辺のバグ、②`nan`＝lr過大や勾配爆発（lr低下・クリッピング・ウォームアップ・LayerNorm）、③1バッチ過学習＝丸暗記すらできなければ実装バグがあると数秒で切り分けられる、の3点をまとめます。
- **解答の目安**：①損失がまったく下がらないときは、学習率が小さすぎるか、`zero_grad()`・`requires_grad` 忘れや勾配が切れているバグを疑い、学習率を上げてコードを見直します。②途中で `nan` になったときは学習率が大きすぎるか勾配爆発が原因なので、学習率を下げ、勾配クリッピングやウォームアップ・LayerNormを入れます。また本番前に1バッチだけで過学習させて訓練損失がほぼ0まで下がるか確認すれば、「丸暗記すらできない＝モデルや学習ループにバグがある」ことを数秒で見抜けます。

**考え方**：「診察表」と「1バッチ過学習テスト」を手に入れておけば、第12〜13章で自分でGPTを実装するときも迷わず原因を突き止められます。

1. 症状①（下がらない）：`lr` が小さすぎる、または `zero_grad()` / `backward()` / `step()` 周辺のバグを疑う。
1. 症状②（`nan`）：`lr` 過大・勾配爆発 → `lr` を下げる＋勾配クリッピング＋ウォームアップ＋LayerNorm。
1. 1バッチ過学習テスト：極小データで損失が0近くまで落ちなければ、ハイパーパラメータ以前に実装バグがあると即座に切り分けられる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/10-training-tips.html#ex-ch10-q15

</details>

**✅ できたらチェック**

- [ ] 最初の損失を $\log V$ と比べる習慣がついた
- [ ] 平均・分散・標準偏差をテンソルで計算できた
- [ ] 重みの大きさで、層を通る数が消えたり飽和したりするのを見た
- [ ] LayerNormを手計算と `nn.LayerNorm` で確かめた
- [ ] AdamWで学習し、過学習の学習曲線を見た
- [ ] ドロップアウト・重み減衰・スケジュールで検証損失を3.18前後まで下げた
- [ ] モデルをGoogleドライブに保存し、読み込めた

> **📝 まとめ**
>
> - まず **最初の損失** を見る。$\log V$ から外れていたら初期化を疑う
> - 重みの大きさが不適切だと、深い層で信号が **消える・飽和する**。$1/\sqrt{\text{入力数}}$ が目安
> - **LayerNorm** は各データの数を平均0・標準偏差1にそろえる（偏差値）
> - **AdamW** は慣性とパラメータごとの歩幅で、SGDより速く賢く下る
> - **過学習** には、ドロップアウト・重み減衰・データを増やす
> - **ウォームアップ＋コサイン減衰**、**勾配クリッピング** で学習を安定させる

**ボンゴ先生**：第2部、完走おめでとう！ ここまでで、ニューラル言語モデルを作って鍛えるための道具は、ほとんどそろった。足りないのは、あと1つだけ。「直前8文字」の窓を、もっと賢く、もっと長く使う方法だ。それが **Attention**。GPTの心臓部だ。第3部で、いよいよGPTを組み立てるぞ。

## 学習のふりかえり・次のステップ

- **困ったとき・エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **忘れる前に定着を確認**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html) で確認問題・類題・7段階の進捗を記録できます
- **この章のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/10-training-tips.html
- **次の章**：[第11章 注目する仕組み：Self-Attention](https://sakunokyumei.github.io/llm-research-textbook/11-attention.html)